# Sports Sentiment Data via RSS (Demo)

### Install Required Libraries




In [ ]:
!pip -q install feedparser pandas python-dateutil


### Import Core Libraries

This cell imports all core dependencies used throughout the pipeline

In [ ]:
import time, re, hashlib
import feedparser
import pandas as pd
from datetime import datetime, timezone
from dateutil import parser as dtparser


### Configure RSS Feed Sources

This cell defines the set of RSS feeds to monitor.

You can freely:
- Add new feeds  
- Remove feeds  
- Rename sources  

This is just a demo using an RSS feed—you’ll need to add additional RSS sources/links for broader sports events coverage.

In [ ]:
# Cell — RSS sources (add/remove freely)
# Cell — Working starter RSS feeds (NBA + NFL + general)
FEEDS = {
    # General / multi-sport
    "ESPN": "https://www.espn.com/espn/rss/news",
    "CBS_SPORTS": "https://www.cbssports.com/rss/headlines/",
    "YAHOO_SPORTS": "https://sports.yahoo.com/rss/",

    # NFL
    "NFL_NEWS": "https://www.nfl.com/rss/rsslanding?searchString=news",
    "ESPN_NFL": "https://www.espn.com/espn/rss/nfl/news",

    # NBA
    "ESPN_NBA": "https://www.espn.com/espn/rss/nba/news",
    "NBA_RUMORS_HOOPSHYPE": "https://hoopshype.com/feed/",

}

print("Feeds configured:", len(FEEDS))
list(FEEDS.items())[:3]


print("Feeds configured:", len(FEEDS))
list(FEEDS.keys())[:10]


Feeds configured: 7
Feeds configured: 7


['ESPN',
 'CBS_SPORTS',
 'YAHOO_SPORTS',
 'NFL_NEWS',
 'ESPN_NFL',
 'ESPN_NBA',
 'NBA_RUMORS_HOOPSHYPE']

###  Core RSS Fetch Utilities

Defines helpers to:
- Parse RSS timestamps safely (UTC)
- Create a unique fingerprint per item (dedupe)
- Fetch all feeds once into a clean DataFrame (sorted, deduped)

Also runs a quick one-time fetch to preview results and total items.


In [ ]:
def safe_parse_dt(x):
    if not x:
        return None
    try:
        d = dtparser.parse(x)
        if not d.tzinfo:
            d = d.replace(tzinfo=timezone.utc)
        return d.astimezone(timezone.utc)
    except Exception:
        return None

def item_fingerprint(title, link, published):
    base = (title or "") + "|" + (link or "") + "|" + (published.isoformat() if published else "")
    return hashlib.sha1(base.encode("utf-8", errors="ignore")).hexdigest()

def fetch_feeds_once(feeds: dict, timeout=15):
    rows = []
    for name, url in feeds.items():
        if not url or "PASTE_" in url:
            continue
        try:
            f = feedparser.parse(url, agent="colab-rss-sports-feed/1.0")
            for e in f.entries:
                title = getattr(e, "title", None)
                link = getattr(e, "link", None)
                summary = getattr(e, "summary", None) or getattr(e, "description", None)

                published_raw = getattr(e, "published", None) or getattr(e, "updated", None)
                published_dt = safe_parse_dt(published_raw)

                fp = item_fingerprint(title, link, published_dt)

                rows.append({
                    "source": name,
                    "published_utc": published_dt,
                    "title": title,
                    "link": link,
                    "summary": summary,
                    "fp": fp,
                })
        except Exception as ex:
            rows.append({
                "source": name,
                "published_utc": None,
                "title": None,
                "link": None,
                "summary": f"ERROR: {ex}",
                "fp": f"error:{name}",
            })
        time.sleep(0.15)  # polite throttle

    df = pd.DataFrame(rows)
    if len(df):
        # Sort newest first; some feeds have missing published times
        df = df.sort_values(["published_utc"], ascending=False, na_position="last")
        df = df.drop_duplicates(subset=["fp"], keep="first")
    return df

df_rss = fetch_feeds_once(FEEDS)
display(df_rss.head(30))
print("Total items:", len(df_rss))


,source,published_utc,title,link,summary,fp
65,YAHOO_SPORTS,2026-01-26 04:45:33+00:00,Las Vegas Raiders and the Buffalo Bills are in...,https://sports.yahoo.com/articles/las-vegas-ra...,With the Broncos eliminated from the postseaso...,b8d84f64683f1644e5becad3a58996cbb7f0ceb1
66,YAHOO_SPORTS,2026-01-26 04:43:35+00:00,Photos: Best images from Thunder's 103-101 los...,https://sports.yahoo.com/articles/photos-best-...,The best photos from the Oklahoma City Thunder...,11bbd167da9db5d889a9e5a16a6702cfece76ff9
68,YAHOO_SPORTS,2026-01-26 04:43:00+00:00,Super Bowl Prop Bets: Odds and Expert Picks fo...,https://sports.yahoo.com/articles/super-bowl-p...,We share the most popular Super Bowl props and...,86b2e26eda383167ffd1f3b50650251976e772ef
67,YAHOO_SPORTS,2026-01-26 04:43:00+00:00,Super Bowl National Anthem Length: Charlie Put...,https://sports.yahoo.com/articles/super-bowl-n...,A look at Over/Under odds for the National Ant...,db30bfe312fa7e2ecd264b572622161b07504e53
69,YAHOO_SPORTS,2026-01-26 04:41:49+00:00,McVay bristles at question about Stafford's fu...,https://sports.yahoo.com/articles/mcvay-bristl...,The Los Angeles Rams quarterback turns 38 in t...,8fbd9e9d7b8c54c6b0225f1dc149810e03249e57
70,YAHOO_SPORTS,2026-01-26 04:41:35+00:00,Writer suggests 'reasonable' trade package for...,https://sports.yahoo.com/articles/writer-sugge...,Would it be worth it for the Lakers to give up...,75147f0cd04952aee5fec4c094ab8e00f7184fae
71,YAHOO_SPORTS,2026-01-26 04:40:55+00:00,Dru Smith keeps impressing with his defense 'O...,https://sports.yahoo.com/articles/dru-smith-ke...,Dru Smith has continued to be one of the stand...,9849427cefb85bd8c8d1eab7fccc1fdf3952cff5
72,YAHOO_SPORTS,2026-01-26 04:40:43+00:00,NFL fans raise conspiracy theory over Super Bo...,https://sports.yahoo.com/articles/nfl-fans-rai...,Some people are buried deep into NFL conspirac...,03963c4e13d9a55d1f218e9888f1d1a29aadca77
73,YAHOO_SPORTS,2026-01-26 04:40:08+00:00,Russell Wilson praises Sam Darnold for 'inspir...,https://sports.yahoo.com/articles/russell-wils...,Here's what Wilson said after the Seahawks' win.,b60ea2b179230ceac0d541903c494c1a052fafc6
74,YAHOO_SPORTS,2026-01-26 04:38:56+00:00,Sean McVay left at a loss for words after Rams...,https://sports.yahoo.com/articles/sean-mcvay-l...,Rams coach Sean McVay had an honest reaction t...,e0f2448a82a36f6c2d6f3dd264bb0a019772f58d


Total items: 142


###  Live RSS Polling (Clean Output)

Continuously polls the feeds and prints only new items in a compact format:

`[time UTC] sport  topic  source  title`
`  link`

Auto-stops after a fixed runtime, you can adjust the time for RSS Polling based on your preference


In [ ]:
# ================================
# RSS clean polling
# Requires: FEEDS dict + fetch_feeds_once(FEEDS) already defined
# ================================

import time
from datetime import datetime, timezone

MAX_RUNTIME_MINUTES = 10   # auto stop after N minutes
POLL_INTERVAL = 90         # poll interval (seconds)

# --- guard ---
if "FEEDS" not in globals() or FEEDS is None or not isinstance(FEEDS, dict) or len(FEEDS) == 0:
    raise NameError("FEEDS is not defined. Run the FEEDS cell first (the one that prints 'Feeds configured: ...').")

if "fetch_feeds_once" not in globals():
    raise NameError("fetch_feeds_once() is not defined. Run the earlier cell that defines it first.")

def _fmt_utc(dt):
    if dt is None:
        return "??:??:?? UTC"
    if hasattr(dt, "tzinfo") and dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc).strftime("%H:%M:%S UTC")

def _classify_sport_and_topic(text: str):
    """
    Very lightweight keyword tagging to match your screenshot style.
    Returns: (sport_tag, topic_tag)
    """
    if not text:
        text = ""
    t = text.lower()

    sport_tags = []
    if any(k in t for k in ["nfl", "steelers", "patriots", "broncos", "quarterback", "qb", "touchdown", "super bowl"]):
        sport_tags.append("NFL")
    if any(k in t for k in ["nba", "warriors", "lakers", "suns", "timberwolves", "nbpa"]):
        sport_tags.append("NBA")
    if not sport_tags:
        sport = "GENERAL"
    else:
        sport = ",".join(dict.fromkeys(sport_tags))  # keep order, unique

    # topic
    if any(k in t for k in ["injury", "out", "sprain", "surgery", "fracture", "questionable", "ruled out"]):
        topic = "injury"
    elif any(k in t for k in ["lineup", "starting", "starter", "inactive", "roster", "depth chart"]):
        topic = "lineup"
    elif any(k in t for k in ["trade", "sign", "release", "waive", "acquire", "contract"]):
        topic = "trade"
    elif any(k in t for k in ["game", "vs.", "championship", "playoff", "final", "score", "record"]):
        topic = "game_event"
    else:
        topic = "news"

    return sport, topic

start_time = time.time()
seen_fp = set()

print(f"🛰️  RSS clean polling started. every {POLL_INTERVAL}s (auto-stops after {MAX_RUNTIME_MINUTES} min)")

while True:
    df = fetch_feeds_once(FEEDS)

    if df is not None and len(df):
        # only new items
        if "fp" in df.columns:
            df_new = df[~df["fp"].isin(seen_fp)].copy()
        else:
            df_new = df.copy()

        # sort oldest -> newest so it reads naturally like a feed
        if "published_utc" in df_new.columns:
            df_new = df_new.sort_values("published_utc", ascending=True, na_position="last")

        # print rows
        for _, row in df_new.iterrows():
            src   = row.get("source", "")
            title = row.get("title", "") or ""
            link  = row.get("link", "") or ""
            pub   = row.get("published_utc", None)
            summ  = row.get("summary", "") or ""

            text_for_tags = f"{title} {summ}"
            sport_tag, topic_tag = _classify_sport_and_topic(text_for_tags)

            print(f"[{_fmt_utc(pub)}] {sport_tag:<6} {topic_tag:<9} {src:<12} {title}")
            if link:
                print(f"  {link}")

            if "fp" in row and row["fp"]:
                seen_fp.add(row["fp"])

    # auto-stop
    if (time.time() - start_time) / 60 >= MAX_RUNTIME_MINUTES:
        print("\nReached max runtime. Stopping RSS polling automatically.")
        break

    time.sleep(POLL_INTERVAL)


🛰️  RSS clean polling started. every 90s (auto-stops after 10 min)
[09:34:54 UTC] GENERAL news      ESPN_NBA     Keep, cut or add? What to do with George, Markkanen and six other trending players
  https://www.espn.com/fantasy/basketball/story/_/id/47701635/espn-fantasy-basketball-nba-risers-fallers-keep-cut-add
[12:42:02 UTC] NFL,NBA news      ESPN         'You should've never been born': How Buss family infighting drove the $10B sale of the Lakers
  https://www.espn.com/nba/story/_/id/47594947/inside-jeanie-jerry-buss-family-infighting-drove-10b-sale-los-angeles-lakers-mark-walter
[10:44:26 UTC] NFL    news      ESPN_NFL     Ranking the NFL's best offensive, defensive coordinator openings, plus intel on top candidates
  https://www.espn.com/nfl/story/_/id/47674143/2026-nfl-offseason-rankings-offensive-defensive-coordinator-openings-intel
[13:17:21 UTC] NFL    lineup    ESPN_NFL     Steelers to hire Mike McCarthy: Six questions on the new coach, plus a grade
  https://www.espn.com/nfl